# Resolução 01 — Primeiros dados ambientais

**Curso:** Tecnologia, Dados e IA na Gestão Ambiental
**Encontros:** 2, 3, 4 e 8

Este é o **notebook resolvido** do encontro de primeiros dados. Ele roda do início ao fim
com **Runtime → Run all**: os dados são baixados automaticamente do repositório.

O que você vai encontrar aqui:

1. leitura e limpeza de um CSV com pandas;
2. estatísticas descritivas com resposta comentada;
3. gráfico de temperatura e gráfico de chuva;
4. comparação entre dias secos e chuvosos;
5. teste de valores atípicos com IQR;
6. modelo de ficha de fonte preenchido;
7. código comentado para trocar pelo CSV real do INMET.

> Os dados são **sintéticos e didáticos**. Não representam medições reais do INMET.

## Passo 0 — Preparar o ambiente e baixar os dados

A célula abaixo baixa o CSV direto do GitHub. Se o arquivo já existir na sessão, ela apenas
reaproveita. Isso funciona no Colab e também localmente.

In [ ]:
from pathlib import Path
import urllib.request

BASE_DADOS = "https://raw.githubusercontent.com/Marquezbertin/Curso-Tecnologia-Dados-Gestao-Ambiental/main/06-DADOS/exemplos"

def baixar(nome, base=BASE_DADOS):
    destino = Path(nome)
    if not destino.exists():
        print(f"Baixando {nome}...")
        urllib.request.urlretrieve(f"{base}/{nome}", destino)
    print(f"Arquivo pronto: {destino.resolve()}")
    return destino

ARQUIVO = baixar("exemplo-inmet-sintetico.csv")

## Passo 1 — Ler e conhecer a tabela

Perguntas que todo analista faz antes de calcular qualquer coisa:

- quantas linhas e colunas existem?
- qual é o período?
- os tipos estão corretos?
- existem nulos, duplicatas ou códigos de erro?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(ARQUIVO)

print("Linhas:", df.shape[0], "| Colunas:", df.shape[1])
print("Colunas:", df.columns.tolist())
df.head()

In [ ]:
df.info()

In [ ]:
print("Duplicatas:", int(df.duplicated().sum()))
print("Valores nulos por coluna:")
print(df.isna().sum().to_string())
print("\nCódigos -9999 (falha de medição):")
print((df.select_dtypes("number") == -9999).sum().to_string())

## Passo 2 — Limpar

A limpeza mínima é: converter data, tratar `-9999`/nulos, ordenar e conferir o período.
Não apague linhas sem registrar o critério.

In [ ]:
df["data"] = pd.to_datetime(df["data"])
df = df.replace(-9999, pd.NA)
df = df.dropna().sort_values("data").reset_index(drop=True)

print("Período:", df["data"].min().date(), "a", df["data"].max().date())
print("Dias válidos:", len(df))
print("Nulos restantes:", int(df.isna().sum().sum()))
df.head()

## Passo 3 — Resumo com respostas

A célula calcula o resumo. Depois dela, a interpretação está escrita em texto.

In [ ]:
resumo = pd.Series({
    "dias": len(df),
    "temp_max_media": df["temp_max_c"].mean(),
    "temp_max_maxima": df["temp_max_c"].max(),
    "temp_min_minima": df["temp_min_c"].min(),
    "chuva_total_mm": df["chuva_mm"].sum(),
    "dias_com_chuva": int((df["chuva_mm"] > 0).sum()),
    "maior_chuva_dia_mm": df["chuva_mm"].max(),
})
print(resumo.round(2).to_string())

print("\nDia mais quente:")
print(df.loc[df["temp_max_c"].idxmax(), ["data", "temp_max_c"]].to_string(index=False))
print("\nDia mais chuvoso:")
print(df.loc[df["chuva_mm"].idxmax(), ["data", "chuva_mm"]].to_string(index=False))

### Resposta esperada

- **Período:** 01/01/2024 a 29/02/2024, com **60 dias**.
- **Temperatura máxima:** média de **30,24 °C**; maior valor **35,4 °C** em **14/01/2024**.
- **Temperatura mínima:** menor valor **14,4 °C** em **05/02/2024**.
- **Chuva:** total de **253,2 mm**, em **30 dias** com registro maior que zero.
- **Maior chuva em um dia:** **22,2 mm** em **26/01/2024**.
- **Janeiro** concentra **187,3 mm** (6,04 mm/dia) e **fevereiro** apenas **65,9 mm**
  (2,27 mm/dia). A diferença de dias entre os meses precisa entrar na comparação.

## Passo 4 — Gráfico de temperatura

O gráfico abaixo usa uma área sombreada para mostrar a amplitude diária. O ponto mais alto
é anotado automaticamente.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(df["data"], df["temp_max_c"], color="#c0392b", linewidth=2, label="Temperatura máxima")
ax.plot(df["data"], df["temp_min_c"], color="#2b6cb0", linewidth=2, label="Temperatura mínima")
ax.fill_between(df["data"], df["temp_min_c"], df["temp_max_c"], color="#16724a", alpha=0.12)

i_max = df["temp_max_c"].idxmax()
ax.annotate(
    f"Máxima: {df.loc[i_max, 'temp_max_c']:.1f} °C",
    xy=(df.loc[i_max, "data"], df.loc[i_max, "temp_max_c"]),
    xytext=(10, 18),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "#7a1f1f"},
    color="#7a1f1f",
    fontsize=9,
)

ax.set_title("Temperatura diária — dados sintéticos didáticos")
ax.set_xlabel("Data")
ax.set_ylabel("Temperatura (°C)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Leitura do gráfico (resposta comentada)

1. **O que o gráfico mostra:** a variação diária da temperatura máxima e mínima entre janeiro
   e fevereiro de 2024. Há dias mais quentes no fim de janeiro e início de fevereiro.
2. **Valor atípico:** pelo critério de IQR (calculado abaixo), **não há outlier**. O valor de
   35,4 °C é o máximo, mas ainda está dentro do limite superior de 38,25 °C.
3. **O que o gráfico não mostra:** a temperatura ao longo do dia, a localização e o erro da
   estação, a umidade, o vento e as causas do padrão. Também não mostra se a série é
   representativa de outros anos.
4. **Pergunta nova:** "A queda de temperatura nos dias de chuva é consistente ou foi
   coincidência desta série curta?"

## Passo 5 — Chuva e comparação entre dias secos e chuvosos

Aqui a pergunta é: **dias com chuva são mais frios?** A resposta precisa de um grupo de
comparação, não de uma impressão visual.

In [ ]:
df["choveu"] = df["chuva_mm"] > 0

comparacao = df.groupby("choveu").agg(
    dias=("data", "count"),
    temp_max_media=("temp_max_c", "mean"),
    temp_min_media=("temp_min_c", "mean"),
    chuva_media=("chuva_mm", "mean"),
).round(2)

comparacao.index = ["sem chuva", "com chuva"]
comparacao

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(df["data"], df["chuva_mm"], color="#2b6cb0", width=0.8)
ax.set_title("Chuva diária — dados sintéticos didáticos")
ax.set_xlabel("Data")
ax.set_ylabel("Chuva (mm)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Resposta

- Dias **sem chuva:** 30 dias, temperatura máxima média de **30,02 °C**.
- Dias **com chuva:** 30 dias, temperatura máxima média de **30,46 °C**.
- A diferença é de apenas **0,44 °C** — pequena e compatível com variação aleatória nesta
  amostra.
- A correlação entre temperatura máxima e chuva é **0,183**, fraca. Isso **não** prova que
  chuva e temperatura são independentes; apenas indica que, nesta série curta, a relação
  linear é fraca.
- **Cuidado:** não transforme "dias chuvosos têm média ligeiramente maior" em "chuva esquenta".
  Correlação não é causalidade, e a amostra é pequena.

## Passo 6 — Valores atípicos com IQR

O critério: um valor é atípico se estiver abaixo de `Q1 - 1,5 × IQR` ou acima de
`Q3 + 1,5 × IQR`.

In [ ]:
q1, q3 = df["temp_max_c"].quantile([0.25, 0.75])
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

atipicos = df[(df["temp_max_c"] < limite_inferior) | (df["temp_max_c"] > limite_superior)]

print(f"Q1 = {q1:.2f} °C | Q3 = {q3:.2f} °C | IQR = {iqr:.2f} °C")
print(f"Limites: {limite_inferior:.2f} °C a {limite_superior:.2f} °C")
print(f"Valores atípicos: {len(atipicos)}")

if len(atipicos):
    print(atipicos[["data", "temp_max_c"]].to_string(index=False))

**Resposta:** Q1 = 28,25 °C, Q3 = 32,25 °C, IQR = 4,00 °C. Limites: 22,25 °C a 38,25 °C.
Nenhum dia ficou fora dos limites. O máximo de 35,4 °C é alto, mas não é atípico por este
critério. Em uma série real, um valor assim merece conferência na fonte antes de ser aceito.

## Passo 7 — Desafio: trocar pelo CSV real do INMET

O código abaixo **não falha** se você ainda não tiver o arquivo: ele apenas mostra as
instruções. Para usar o desafio, baixe o CSV da estação do seu município em
<https://portal.inmet.gov.br/>, coloque na sessão do Colab, defina `ARQUIVO_REAL` e rode.

O formato do INMET muda entre estações. O importante é mapear as colunas para:
`data`, `temp_max_c`, `temp_min_c`, `chuva_mm`.

In [ ]:
ARQUIVO_REAL = None  # exemplo: "meu-municipio.csv"

if ARQUIVO_REAL:
    bruto = pd.read_csv(
        ARQUIVO_REAL,
        sep=";",
        skiprows=8,
        decimal=",",
        na_values=["-9999"],
        encoding="latin-1",
    )
    bruto.columns = [c.strip() for c in bruto.columns]
    print("Colunas encontradas:")
    print(bruto.columns.tolist())
    print("\nConfira o cabeçalho e ajuste o dicionário de colunas antes de continuar.")
else:
    print("Desafio opcional ainda não configurado.")
    print("1. Baixe o CSV da estação em https://portal.inmet.gov.br/")
    print("2. Faça upload no Colab")
    print("3. Defina ARQUIVO_REAL = 'nome-do-arquivo.csv'")
    print("4. Rode novamente e mapeie as colunas para data, temp_max_c, temp_min_c, chuva_mm")

### Como registrar a resposta do desafio

Preencha a ficha abaixo no seu caderno ou no relatório do grupo:

| Campo | Resposta |
|---|---|
| Fonte | Exemplo sintético do repositório (não é INMET real) |
| Estação | Não se aplica ao arquivo sintético |
| Período | 01/01/2024 a 29/02/2024 (60 dias) |
| Variáveis | temperatura máxima, temperatura mínima e chuva |
| Limitação | Série curta, sintética, uma localidade, sem hora nem erro de medição |
| Uso recomendado | Praticar leitura e limpeza de dados |
| Uso não recomendado | Decisão ambiental real, laudo ou comparação histórica |

Se você usar dados reais, troque a linha "Fonte" pelo nome da estação, o período e a data de
acesso. Sem isso, o gráfico não é evidência auditável.

## Passo 8 — Para ir além (com respostas)

### Exercício A — Qual mês choveu mais?

O código abaixo responde. Compare a **soma** e também a **média por dia**, porque janeiro tem
31 dias e fevereiro tem 29.

In [ ]:
df["mes"] = df["data"].dt.month
por_mes = df.groupby("mes").agg(
    dias=("data", "count"),
    chuva_total=("chuva_mm", "sum"),
    temp_max_media=("temp_max_c", "mean"),
)
por_mes["chuva_por_dia"] = por_mes["chuva_total"] / por_mes["dias"]
por_mes.round(2)

**Resposta:** janeiro teve **187,3 mm** em 31 dias (6,04 mm/dia); fevereiro teve **65,9 mm**
em 29 dias (2,27 mm/dia). Janeiro foi mais chuvoso tanto no total quanto na média diária.

### Exercício B — Escreva uma conclusão com limite

**Exemplo de resposta aceitável:**

> "Nesta série sintética de 60 dias, a temperatura máxima ficou entre 24,3 °C e 35,4 °C, com
> média de 30,2 °C. A chuva se concentrou em janeiro. A relação entre chuva e temperatura foi
> fraca (r = 0,183). Como a série é curta e sintética, não é possível concluir tendência
> climática nem generalizar para o município."

Uma conclusão com limite nomeia **o que foi medido**, **o período**, **o resultado** e
**o que não se pode concluir**. É isso que a rubrica cobra.

## Checklist de entrega

- [ ] O notebook rodou inteiro com **Run all**.
- [ ] O período e o número de dias estão escritos.
- [ ] A limpeza registrou o que foi removido e por quê.
- [ ] O gráfico tem título, eixos e unidade.
- [ ] A leitura responde ao que o gráfico mostra e ao que ele não mostra.
- [ ] A ficha de fonte está preenchida.
- [ ] A conclusão não confunde correlação com causalidade.

**Próximo notebook resolvido:** [`02-resolucao-mapas-e-territorio.ipynb`](02-resolucao-mapas-e-territorio.ipynb)